In [1]:
import pandas as pd

reg = pd.read_csv("data/csv/raw/regimenes.csv", parse_dates=["date"], index_col="date")

print("Columnas:", reg.columns.tolist())
print(f"Filas: {len(reg)}  ({reg.index.min().date()} -> {reg.index.max().date()})")
print()

# Reparto de regímenes (debería ser ~Acumulación 39% / Bajista 36% / Alcista 25%)
col = "regimen" if "regimen" in reg.columns else "estado_hmm"
print("Reparto de regímenes:")
print(reg[col].value_counts())
print()
print("En porcentaje:")
print((reg[col].value_counts(normalize=True) * 100).round(1))
print()

# Cuántos cambios de régimen hay (deberían ser ~27-28)
cambios = (reg[col] != reg[col].shift()).sum()
print(f"Número de tramos (cambios de régimen): {cambios}")
print()

# El régimen de los últimos días
print("Últimos 5 días:")
print(reg[[col]].tail())

Columnas: ['precio', 'vol_30d', 'cum_ret_60d', 'dist_sma200', 'drawdown', 'fg', 'estado_hmm', 'regimen']
Filas: 2847  (2018-08-19 -> 2026-06-04)

Reparto de regímenes:
regimen
Acumulacion    1116
Bajista        1029
Alcista         702
Name: count, dtype: int64

En porcentaje:
regimen
Acumulacion    39.2
Bajista        36.1
Alcista        24.7
Name: proportion, dtype: float64

Número de tramos (cambios de régimen): 28

Últimos 5 días:
                regimen
date                   
2026-05-31  Acumulacion
2026-06-01  Acumulacion
2026-06-02  Acumulacion
2026-06-03  Acumulacion
2026-06-04  Acumulacion


In [1]:
import torch
ckpt = torch.load("models/lstm_final.pt", map_location="cpu", weights_only=False)
print("CLAVES:", list(ckpt.keys()))
print("arquitectura:", ckpt.get("arquitectura"))
print("seq_len:", ckpt.get("seq_len"), "| horizon:", ckpt.get("horizon"))
print("n feature_cols:", len(ckpt.get("feature_cols", [])))
print("n regime_cols:", len(ckpt.get("regime_cols", [])))
# y las formas reales de los pesos:
for k, v in ckpt["state_dict"].items():
    print(k, tuple(v.shape))

CLAVES: ['state_dict', 'arquitectura', 'feature_cols', 'regime_cols', 'seq_len', 'horizon', 'target_col']
arquitectura: {'n_features': 19, 'hidden_sizes': (48, 24), 'horizon': 3, 'dropout': 0.35}
seq_len: 30 | horizon: 3
n feature_cols: 16
n regime_cols: 3
lstms.0.weight_ih_l0 (128, 19)
lstms.0.weight_hh_l0 (128, 32)
lstms.0.bias_ih_l0 (128,)
lstms.0.bias_hh_l0 (128,)
head.0.weight (16, 32)
head.0.bias (16,)
head.3.weight (3, 16)
head.3.bias (3,)


In [2]:
import torch

# 1. Cargar el checkpoint actual
ckpt = torch.load("models/lstm_final.pt", map_location="cpu", weights_only=False)

# 2. Deducir la arquitectura REAL desde los pesos (la verdad)
state = ckpt["state_dict"]
n_features = state["lstms.0.weight_ih_l0"].shape[1]
n_capas = sum(1 for k in state if k.endswith(".weight_ih_l0"))
hidden_sizes = tuple(state[f"lstms.{i}.weight_ih_l0"].shape[0] // 4 for i in range(n_capas))
horizon = state["head.3.weight"].shape[0]

print("ANTES (corrupto):", ckpt["arquitectura"])

# 3. Corregir el campo arquitectura
ckpt["arquitectura"] = {
    "n_features": n_features,
    "hidden_sizes": hidden_sizes,
    "horizon": horizon,
    "dropout": ckpt["arquitectura"].get("dropout", 0.35),  # el dropout no afecta a la forma
}

print("DESPUÉS (correcto):", ckpt["arquitectura"])

# 4. Guardar (hago una copia de seguridad antes, por si acaso)
import shutil
shutil.copy("models/lstm_final.pt", "models/lstm_final_backup.pt")
torch.save(ckpt, "models/lstm_final.pt")
print("✓ Guardado. Backup en lstm_final_backup.pt")

ANTES (corrupto): {'n_features': 19, 'hidden_sizes': (48, 24), 'horizon': 3, 'dropout': 0.35}
DESPUÉS (correcto): {'n_features': 19, 'hidden_sizes': (32,), 'horizon': 3, 'dropout': 0.35}
✓ Guardado. Backup en lstm_final_backup.pt
